# 3.2 도구 사용(Tool Using)

도구 사용은 에이전트가 문제 해결을 위해 외부 도구를 선택하고 실행하는 능력이다.

### 핵심 패턴: ReAct

**Reasoning + Acting**의 결합:
```
Thought(생각) → Action(행동) → Observation(관찰)
```

- **Thought**: "다음에 무엇을 할지 생각한다."
- **Action**: "도구를 호출한다."
- **Observation**: "결과를 분석한다."

## 도구 선택과 실행

### 도구 선택 정책

어떤 도구를 사용할지 결정:
```
도구 = LLM(작업 설명 + 이용가능 도구 목록)
```

### 도구 체이닝 (Tool Composition)

여러 도구를 순차 사용:
```
도구1의 출력 → 도구2의 입력 → 도구3 ...
```

### 오류 복구

도구 실행 실패 시 대체 도구 선택:
```
실패 → 대체 도구 제안 → 재시도
```

## 환경 설정

In [1]:
from utils_openai import *

heading("3.2 도구 사용(Tool Using)")
step_print(1, "도구 정의", [
    "계산기: 수학 계산 수행",
    "검색: 정보 조회",
    "메모리: 저장된 정보 검색"
])


────────────────────────────────────────
  3.2 도구 사용(Tool Using)
────────────────────────────────────────
  [Step 1] 도구 정의
    • 계산기: 수학 계산 수행
    • 검색: 정보 조회
    • 메모리: 저장된 정보 검색


## 실습 1: 도구 정의 및 실행

In [3]:
# 간단한 도구들
def calculator(expression: str) -> str:
    """수식을 계산한다."""
    try:
        result = eval(expression)
        return f"계산 결과: {result}"
    except:
        return "계산 오류: 유효하지 않은 수식이다"

def search_info(query: str) -> str:
    """정보를 검색한다 (시뮬레이션)."""
    database = {
        "파이썬": "파이썬은 간단하고 읽기 쉬운 프로그래밍 언어다.",
        "강화학습": "강화학습은 에이전트가 환경과 상호작용하며 학습한다.",
        "메모리": "메모리는 에이전트가 과거 경험을 저장하고 활용한다."
    }
    return database.get(query, f"'{query}'에 대한 정보가 없다.")

step_print(1, "도구 테스트", "각 도구를 직접 호출한다")

# 도구 테스트
print(f"계산기: {calculator('10 + 5')}")
print(f"검색: {search_info('파이썬')}")

  [Step 1] 도구 테스트
    각 도구를 직접 호출한다
계산기: 계산 결과: 15
검색: 파이썬은 간단하고 읽기 쉬운 프로그래밍 언어다.


## 실습 2: OpenAI Function Calling으로 도구 선택

In [4]:
step_print(2, "도구 스키마 정의", "LLM이 호출할 도구를 정의한다")

# 도구 정의
calc_schema = tool_schema(
    "calculator",
    "수학 계산을 수행한다",
    {"expression": {"type": "string", "description": "계산할 수식"}},
    ["expression"]
)

search_schema = tool_schema(
    "search",
    "정보를 검색한다",
    {"query": {"type": "string", "description": "검색어"}},
    ["query"]
)

print("도구 스키마:")
print(f"  • {calc_schema['function']['name']}: {calc_schema['function']['description']}")
print(f"  • {search_schema['function']['name']}: {search_schema['function']['description']}")

  [Step 2] 도구 스키마 정의
    LLM이 호출할 도구를 정의한다
도구 스키마:
  • calculator: 수학 계산을 수행한다
  • search: 정보를 검색한다


In [7]:
step_print(3, "도구 선택 및 실행", "LLM이 적절한 도구를 선택하여 실행한다")

# 도구 맵
tools_map = {
    "calculator": calculator,
    "search": search_info
}

# 사용자 요청
user_request = "강화학습에 대해 알려주고, 10의 제곱을 계산해줘."

# LLM이 도구를 선택하도록 함
messages = [
    {"role": "user", "content": user_request}
]

response = chat(messages, tools=[calc_schema, search_schema], temperature=0.7, max_tokens=200)

print(f"사용자 요청: {user_request}")

# 도구 호출 실행
if hasattr(response.choices[0].message, 'tool_calls') and response.choices[0].message.tool_calls:
    for tc in response.choices[0].message.tool_calls:
        tool_name = tc.function.name
        tool_args = json.loads(tc.function.arguments)
        
        print(f"선택된 도구: {tool_name}")
        print(f"인자: {tool_args}")
        
        if tool_name in tools_map:
            result = tools_map[tool_name](list(tool_args.values())[0])
            print(f"결과: {result}")
else:
    print(f"LLM 응답: {response.choices[0].message.content}")

  [Step 3] 도구 선택 및 실행
    LLM이 적절한 도구를 선택하여 실행한다
사용자 요청: 강화학습에 대해 알려주고, 10의 제곱을 계산해줘.
선택된 도구: search
인자: {'query': '강화학습'}
결과: 강화학습은 에이전트가 환경과 상호작용하며 학습한다.
선택된 도구: calculator
인자: {'expression': '10^2'}
결과: 계산 결과: 8


## 실습 3: ReAct 루프 구현

In [9]:
heading("ReAct 루프 시연")
step_print(1, "Thought", "문제를 분석하고 다음 행동을 계획한다")

problem = "강화학습의 핵심 개념과 10의 3제곱 값을 알아야 한다."
thought = ask(
    f"""문제: {problem}
이 문제를 해결하기 위해 취할 단계를 3 단계로 설명하다.""",
    temperature=0.6,
    max_tokens=150
)

print(f"분석:{thought}")


────────────────────────────────────────
  ReAct 루프 시연
────────────────────────────────────────
  [Step 1] Thought
    문제를 분석하고 다음 행동을 계획한다
분석:강화학습의 핵심 개념과 10의 3제곱 값을 알아보기 위해 다음과 같은 3단계를 취할 수 있습니다.

### 1단계: 강화학습의 기본 개념 이해하기
- **강화학습(RL, Reinforcement Learning)**은 에이전트가 환경과 상호작용하면서 최적의 행동을 학습하는 기계 학습의 한 분야입니다. 에이전트는 상태(state)를 관찰하고, 행동(action)을 선택하며, 그 행동의 결과로 보상(reward)을 받습니다. 목표는 누적 보상을 최대화하는 정책(policy)을 학습하는 것입니다.
- 핵심 요소로는 **상태


In [11]:
step_print(2, "Action", "도구를 선택하고 실행한다")

# 도구 1: 정보 검색
print("  [도구 1] 검색 실행")
result1 = search_info("강화학습")
print(f"  결과: {result1}")

# 도구 2: 계산 수행
print("[도구 2] 계산 실행")
result2 = calculator("10 ** 3")
print(f"  결과: {result2}")

  [Step 2] Action
    도구를 선택하고 실행한다
  [도구 1] 검색 실행
  결과: 강화학습은 에이전트가 환경과 상호작용하며 학습한다.
[도구 2] 계산 실행
  결과: 계산 결과: 1000


In [13]:
step_print(3, "Observation", "결과를 종합하여 최종 답변을 생성한다")

observation = f"""
도구 실행 결과:
1. 강화학습: {result1}
2. 계산: {result2}

위 결과를 종합하여 최종 답변을 구성하다.
"""

final_answer = ask(
    observation,
    system="당신은 도구 실행 결과를 종합하는 전문가다.",
    temperature=0.6,
    max_tokens=150
)

print(f"최종 답변:{final_answer}")

  [Step 3] Observation
    결과를 종합하여 최종 답변을 생성한다
최종 답변:강화학습은 에이전트가 환경과 상호작용하며 학습하는 과정으로, 이 과정에서 에이전트는 다양한 행동을 시도하고 그 결과를 통해 최적의 행동 전략을 개발합니다. 이러한 학습 과정은 수많은 계산을 포함하며, 예를 들어 특정 문제에 대한 계산 결과가 1000이라는 수치를 도출할 수 있습니다. 따라서, 강화학습은 환경에서의 경험을 통해 에이전트가 최적의 결정을 내리는 데 필요한 계산적 결과들을 활용하는 중요한 방법론입니다.


## 실습 4: 도구 체이닝 (여러 도구 순차 사용)

In [14]:
heading("도구 체이닝")

# 3단계 도구 체이닝
step_print(1, "검색", "기본 정보를 검색한다")
info = search_info("메모리")
print(f"결과: {info}")

step_print(2, "계산", "관련된 수치를 계산한다")
calc = calculator("1000 * 0.8")  # 메모리의 80% 사용
print(f"결과: {calc}")

step_print(3, "최종화", "결과를 종합한다")
final = f"{info} 최대 메모리의 80%를 활용할 수 있다 (약 800MB)."
print(f"결과: {final}")


────────────────────────────────────────
  도구 체이닝
────────────────────────────────────────
  [Step 1] 검색
    기본 정보를 검색한다
결과: 메모리는 에이전트가 과거 경험을 저장하고 활용한다.
  [Step 2] 계산
    관련된 수치를 계산한다
결과: 계산 결과: 800.0
  [Step 3] 최종화
    결과를 종합한다
결과: 메모리는 에이전트가 과거 경험을 저장하고 활용한다. 최대 메모리의 80%를 활용할 수 있다 (약 800MB).


## 핵심 개념

### ReAct 루프

```
문제 → [Thought] 생각 → [Action] 행동 → [Observation] 관찰 → 답변
```

### 도구 선택 공식

```
선택된_도구 = LLM(작업 + 도구_목록)
```

### 도구 체이닝

```
결과 = 도구1 → 도구2 → 도구3
```

각 도구의 출력이 다음 도구의 입력이 된다.